In [1]:
#import labs


import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# MULTI-CLASS IMAGE CLASSIFICATION USING RESNET50
# TRAIN + VALIDATION + TEST
# TEST ACCURACY + CONFUSION MATRIX + CLASSIFICATION REPORT


# DATASET PATH
TRAIN_PATH = "/kaggle/input/datasets/misrakahmed/vegetable-image-dataset/Vegetable Images/train"
VAL_PATH   = "/kaggle/input/datasets/misrakahmed/vegetable-image-dataset/Vegetable Images/validation"
TEST_PATH  = "/kaggle/input/datasets/misrakahmed/vegetable-image-dataset/Vegetable Images/test"


#base congif
IMG_SIZE = (224,224)
BATCH_SIZE = 32
EPOCHS = 30
MODEL_PATH = "best_resnet.weights.h5".

SyntaxError: invalid syntax (1902822431.py, line 16)

In [5]:

# IMAGE AUGMENTATION
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

# LOAD DATA
train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = test_datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/kaggle/input/datasets/misrakahmed/vegetable-image-dataset/Vegetable Images/train'

In [ ]:



# CLASS INFO
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print("Classes:", class_names)

# ===============================================================
# DISPLAY ONE IMAGE FROM EACH CLASS
# ===============================================================
plt.figure(figsize=(15,8))

for i, cls in enumerate(class_names):
    folder = os.path.join(TRAIN_PATH, cls)
    img_name = os.listdir(folder)[0]
    img_path = os.path.join(folder, img_name)

    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img = tf.keras.utils.img_to_array(img)/255.0

    plt.subplot(2, (num_classes+1)//2, i+1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

# LOAD RESNET50
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# ===============================================================
# CUSTOM MODEL
# ===============================================================
inputs = Input(shape=(224,224,3))

x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)

outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)


In [ ]:


# COMPILE
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# CALLBACKS
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    MODEL_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)


In [ ]:

# TRAIN MODEL
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)

In [ ]:


# LOAD BEST WEIGHTS
model.load_weights(MODEL_PATH)

# TEST ACCURACY
test_loss, test_acc = model.evaluate(test_generator)

print("Test Accuracy :", round(test_acc*100,2), "%")
print("Test Loss     :", round(test_loss,4))


In [ ]:

# TRAINING PLOTS
plt.figure(figsize=(14,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:


# PREDICTIONS ON TEST SET
test_generator.reset()

pred_probs = model.predict(test_generator)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes

# CONFUSION MATRIX
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:

# CLASSIFICATION REPORT
print("\nClassification Report:\n")

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names
)

print(report)


In [ ]:

# SINGLE IMAGE PREDICTION
img_path = "sample.jpg"

img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
img = tf.keras.utils.img_to_array(img)/255.0
img = np.expand_dims(img, axis=0)

pred = model.predict(img)

pred_class = class_names[np.argmax(pred)]
confidence = np.max(pred)

print("Predicted Class :", pred_class)
print("Confidence      :", round(confidence*100,2), "%")